<a href="https://colab.research.google.com/github/MusicalManiac/SatelliteDataAI-UOA/blob/main/Lab2_Answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Code to Question 5
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Test (not just validate) — this is our real hold-out accuracy for 2020
tested = test.classify(classifier, 'predicted')

# Function to export data for confusion matrix
def fc_to_lists(fc, classProp, predProp):
    values = fc.aggregate_array(classProp).getInfo()
    preds = fc.aggregate_array(predProp).getInfo()
    return values, preds

# Get predicted vs actual from test set
y_true, y_pred = fc_to_lists(tested, 'Map_remapped', 'predicted')

# Labels for original classes
label_map = {i + 1: valid_classes.get(i).getInfo() for i in range(10)}
label_names = [label_map[i + 1] for i in range(10)]

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=list(range(1, 11)))
report = classification_report(y_true, y_pred, labels=list(range(1, 11)), target_names=[str(l) for l in label_names])

# Overall test accuracy (this is the number we put in the Q5 figure)
test_accuracy_2020 = accuracy_score(y_true, y_pred)

# Pretty-print
print("Confusion Matrix:")
print(pd.DataFrame(cm, index=[f"Actual {l}" for l in label_names],
                       columns=[f"Pred {l}" for l in label_names]))
print("\nClassification Report:")
print(report)
print(f"\nOverall Test Accuracy (2020): {test_accuracy_2020:.3f}")

def get_s2_composite(year, aoi):
    start = f'{year}-01-01'
    end = f'{year}-12-31'
    s2_year = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterBounds(aoi)
               .filterDate(start, end)
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
               .map(mask_s2_clouds)
               .median()
               .clip(aoi))
    return s2_year

years = [2018, 2022, 2024]
classified_by_year = {}

for yr in years:
    s2_yr = get_s2_composite(yr, aoi)
    classified_by_year[yr] = s2_yr.select(bands).classify(classifier)

def ee_image_to_array(image, region, scale=30):
    arr = geemap.ee_to_numpy(image, region=region, scale=scale)
    return arr[:, :, 0]

arrays = {yr: ee_image_to_array(classified_by_year[yr], aoi) for yr in years}

class_colors = ['#006400','#ffbb22','#ffff4c','#f096ff','#fa0000',
                 '#b4b4b4','#f0f0f0','#0064c8','#0096a0','#fae6a0']
cmap = mcolors.ListedColormap(class_colors)
bounds = np.arange(0.5, 11.5, 1)
norm = mcolors.BoundaryNorm(bounds, cmap.N)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, yr in zip(axes, years):
    ax.imshow(arrays[yr], cmap=cmap, norm=norm)
    ax.set_title(f'Wellington Landcover — {yr}', fontsize=13, fontweight='bold')
    ax.axis('off')

patches = [plt.Rectangle((0,0),1,1, color=class_colors[i]) for i in range(10)]
fig.legend(patches, label_names, loc='lower center', ncol=5, frameon=False,
           bbox_to_anchor=(0.5, -0.05))

fig.suptitle(
    f'Random Forest Landcover Classification (trained on 2020 ESA WorldCover labels)\n'
    f'2020 Test Accuracy: {test_accuracy_2020:.1%}',
    fontsize=14, fontweight='bold', y=1.03
)

caption = (
    "Figure: Landcover classifications for 2018, 2022 and 2024, produced by applying a Random Forest\n"
    "classifier — trained exclusively on 2020 Sentinel-2 imagery and 2020 ESA WorldCover labels — to\n"
    f"Sentinel-2 composites from other years. The reported test accuracy ({test_accuracy_2020:.1%}) reflects\n"
    "performance ONLY on held-out 2020 data. Accuracy cannot be stated for 2018, 2022 or 2024 because no\n"
    "ground-truth landcover labels exist for those years, so there is no way to distinguish genuine landcover\n"
    "change from classifier error without independent reference data for each specific year."
)
fig.text(0.5, -0.18, caption, ha='center', va='top', fontsize=9)

plt.tight_layout()
plt.savefig('wellington_landcover_2018_2022_2024.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#Code to question 6
# Code to get you started
import zipfile
import geopandas as gpd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pandas as pd
import numpy.ma as ma

# Upload the ZIP manually using the Colab UI
from google.colab import files
uploaded = files.upload()  # <- Expects a ZIP

# Unzip
with zipfile.ZipFile("lris-lcdb-v50-land-cover-database-version-50-mainland-new-zealand-SHP.zip", 'r') as zip_ref: #<- Check file names
    zip_ref.extractall("lcdb")

# Read shapefile
gdf = gpd.read_file("lcdb/lcdb-v50-land-cover-database-version-50-mainland-new-zealand.shp") #<- Check file names
print(gdf.head())

# Inspect what year columns exist
print([c for c in gdf.columns if 'Class' in c or 'class' in c])
gdf = gpd.read_file("lcdb/lcdb-v50-land-cover-database-version-50-mainland-new-zealand.shp")
# Reproject to WGS84 to match GEE's expected CRS
gdf = gdf.to_crs("EPSG:4326")
print(gdf.shape)

# Great Barrier Island (Aotea) approximate bounding box
gbi_aoi = ee.Geometry.Rectangle([175.30, -36.35, 175.55, -36.05])

gbi_s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(gbi_aoi)
          .filterDate('2018-12-01', '2019-02-28')  # austral summer 18/19
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
          .map(mask_s2_clouds)
          .median()
          .clip(gbi_aoi))

print("gdf exists:", 'gdf' in dir())
print(gdf.shape)

result = geemap.geopandas_to_ee(gdf)
print("Conversion done")
print(type(result))

lcdb_fc = result
print("lcdb_fc assigned")
print(lcdb_fc.size().getInfo())

class_col = 'Class_2018'

lcdb_fc_clipped = lcdb_fc.filterBounds(gbi_aoi)

training_gbi = gbi_s2.select(bands).sampleRegions(
    collection=lcdb_fc_clipped,
    properties=[class_col],
    scale=30,
    geometries=False,
    tileScale=4
)

print("Total samples:", training_gbi.size().getInfo())

training_gbi = training_gbi.randomColumn('rand_sub')
training_gbi = training_gbi.sort('rand_sub').limit(8000)

training_gbi = training_gbi.randomColumn('random')
train_gbi = training_gbi.filter(ee.Filter.lt('random', 0.7))
valid_gbi = training_gbi.filter(ee.Filter.And(ee.Filter.gte('random', 0.7), ee.Filter.lt('random', 0.9)))
test_gbi  = training_gbi.filter(ee.Filter.gte('random', 0.9))

print("Train:", train_gbi.size().getInfo(), "Valid:", valid_gbi.size().getInfo(), "Test:", test_gbi.size().getInfo())

svm_gbi = ee.Classifier.libsvm(kernelType='RBF', gamma=0.5, cost=10).train(
    features=train_gbi,
    classProperty=class_col,
    inputProperties=bands
)

tested_gbi = test_gbi.classify(svm_gbi, 'predicted')

def fc_to_lists(fc, classProp, predProp):
    values = fc.aggregate_array(classProp).getInfo()
    preds = fc.aggregate_array(predProp).getInfo()
    return values, preds

y_true_gbi, y_pred_gbi = fc_to_lists(tested_gbi, class_col, 'predicted')
gbi_labels = sorted(set(y_true_gbi) | set(y_pred_gbi))

cm_gbi = confusion_matrix(y_true_gbi, y_pred_gbi, labels=gbi_labels)
report_gbi = classification_report(y_true_gbi, y_pred_gbi, labels=gbi_labels, zero_division=0)
test_accuracy_gbi = accuracy_score(y_true_gbi, y_pred_gbi)

print(pd.DataFrame(cm_gbi, index=[f"Actual {l}" for l in gbi_labels], columns=[f"Pred {l}" for l in gbi_labels]))
print(report_gbi)
print(f"Overall Test Accuracy: {test_accuracy_gbi:.3f}")

classified_gbi = gbi_s2.select(bands).classify(svm_gbi)

def ee_image_to_array(image, region, scale=30):
    arr = geemap.ee_to_numpy(image, region=region, scale=scale)
    return arr[:, :, 0]

gbi_array = ee_image_to_array(classified_gbi, gbi_aoi, scale=30)


# Simple NDWI-based water mask to exclude ocean before plotting
ndwi = gbi_s2.normalizedDifference(['B3', 'B8']).rename('NDWI')
water_mask = ndwi.lt(0.1)  # non-water pixels only; tune threshold if ocean still leaks through

classified_gbi_masked = classified_gbi.updateMask(water_mask)

gbi_array = ee_image_to_array(classified_gbi_masked, gbi_aoi, scale=60)
gbi_array = gbi_array.astype(float)
gbi_array[gbi_array == 0] = np.nan

print(gbi_array.shape)
print(np.unique(gbi_array[~np.isnan(gbi_array)]))

masked_array = ma.masked_invalid(gbi_array_idx)

fig, ax = plt.subplots(figsize=(10, 10))
ax.set_facecolor('lightblue')
im = ax.imshow(masked_array, cmap=cmap, vmin=0, vmax=n_classes-1)
ax.set_title('Landcover of Great Barrier Island (Aotea), Summer 2018/19', fontsize=14, fontweight='bold')
ax.axis('off')

patches = [plt.Rectangle((0,0),1,1, color=cmap_colors[i]) for i in range(n_classes)]
labels = [class_lookup.get(c, str(c)) for c in unique_classes]
ax.legend(patches, labels, loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=8, title='LCDB Class')

ax.add_artist(ScaleBar(scale, location='lower left'))

ax.annotate('N', xy=(0.95, 0.15), xytext=(0.95, 0.05), xycoords='axes fraction',
            arrowprops=dict(facecolor='black', width=4, headwidth=10),
            ha='center', fontsize=12, fontweight='bold')

caption = (
    f"Data: Sentinel-2 (COPERNICUS/S2_SR_HARMONIZED), Dec 2018–Feb 2019 median composite.\n"
    f"Training labels: LINZ/Manaaki Whenua LCDB v5.0 (2018). Classifier: SVM (RBF kernel, gamma=0.5, cost=10).\n"
    f"Overall test accuracy: {test_accuracy_gbi:.1%} (evaluated on held-out 10% test split, n=754).\n"
    f"Note: open ocean masked using NDWI threshold (NDWI < 0.1) as LCDB does not classify marine areas."
)
fig.text(0.1, -0.05, caption, ha='left', va='top', fontsize=8)

plt.tight_layout()
plt.savefig('great_barrier_island_landcover_2018_masked.png',
            dpi=300, bbox_inches='tight', facecolor='lightblue')
plt.show()